In [4]:
from ultralytics import YOLO
import os
import cv2

YOLO_MODEL = '../checkpoints/model_main.pt' 
LINEMOD_ROOT = '../dataset/Linemod_preprocessed/data/' 
OUTPUT_CROP_FOLDER = 'RGB_Crops' 
CROP_SIZE = (224, 224)
CLASS_IDS = [f"{i:02d}" for i in range(1, 14)]  # '01' to '13'

os.makedirs(OUTPUT_CROP_FOLDER, exist_ok=True)
model = YOLO(YOLO_MODEL)

# === Run YOLO on each image from each class folder ===
for cls_id in CLASS_IDS:
    class_path = os.path.join(LINEMOD_ROOT, cls_id)
    
    # Skip folders that don't exist (e.g., 03 or 07)
    if not os.path.isdir(class_path):
        print(f"[SKIPPED] Missing class folder: {class_path}")
        continue

    rgb_folder = os.path.join(class_path, 'rgb')
    train_file = os.path.join(class_path, 'train.txt')

    if not os.path.exists(train_file):
        print(f"[SKIPPED] Missing train file for {cls_id}")
        continue

    with open(train_file, 'r') as f:
        image_ids = [line.strip() for line in f.readlines() if line.strip()]

    print(f"[{cls_id}] Processing {len(image_ids)} training images...")

    for img_id in image_ids:
        img_path = os.path.join(rgb_folder, f"{img_id}.png")
        image = cv2.imread(img_path)

        if image is None:
            print(f"[WARNING] Cannot read image: {img_path}")
            continue

        results = model(image)

        for det_idx, det in enumerate(results[0].boxes):
            class_idx = int(det.cls.item()) + 1  # YOLO class 0 → LineMOD folder 01

            # Skip missing LineMOD folders
            if class_idx in [3, 7]:
                continue

            # Extract bounding box
            x1, y1, x2, y2 = map(int, det.xyxy[0].tolist())
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)

            # Crop and resize
            cropped = image[y1:y2, x1:x2]
            cropped_resized = cv2.resize(cropped, CROP_SIZE)

            # Save with correct class label
            filename = f"cls{class_idx:02d}_img{img_id}_det{det_idx}.png"
            save_path = os.path.join(OUTPUT_CROP_FOLDER, filename)
            cv2.imwrite(save_path, cropped_resized)

            print(f" Saved: {filename}")

print("\n All YOLO crops saved.")

[01] Processing 186 training images...

0: 480x640 1 ape, 75.8ms
Speed: 3.6ms preprocess, 75.8ms inference, 319.5ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0004_det0.png

0: 480x640 1 ape, 8.7ms
Speed: 2.4ms preprocess, 8.7ms inference, 4.1ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0009_det0.png

0: 480x640 1 ape, 8.0ms
Speed: 2.6ms preprocess, 8.0ms inference, 2.3ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0021_det0.png

0: 480x640 1 ape, 8.9ms
Speed: 1.4ms preprocess, 8.9ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0030_det0.png

0: 480x640 1 ape, 10.4ms
Speed: 1.6ms preprocess, 10.4ms inference, 2.4ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0044_det0.png

0: 480x640 1 ape, 8.6ms
Speed: 1.7ms preprocess, 8.6ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls01_img0048_det0.png

0: 480x640 1 ape, 9.1ms
Speed: 1.5ms prepr

In [7]:
import csv
import yaml

CROP_FOLDER = 'RGB_Crops'
OUTPUT_CSV = 'pose_labels.csv'
LINEMOD_ROOT = '../dataset/Linemod_preprocessed/data/'

def load_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

# Open CSV to write
with open(OUTPUT_CSV, 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    # Header: crop path + translation + flattened 3x3 rotation
    writer.writerow(['crop_path', 'class_id', 'image_id', 'x', 'y', 'z'] + [f'R{i}' for i in range(1, 10)])

    # Loop over each cropped file
    for filename in os.listdir(CROP_FOLDER):
        if not filename.endswith('.png'):
            continue

        try:
            # Extract metadata from filename
            parts = filename.split('_')
            class_id = parts[0][3:]  # e.g., 'cls01' → '01'
            image_id = parts[1][3:]  # e.g., 'img000123' → '000123'

            # Load corresponding gt.yml file
            gt_path = os.path.join(LINEMOD_ROOT, class_id, 'gt.yml')
            if not os.path.exists(gt_path):
                print(f"[SKIP] Missing gt.yml for class {class_id}")
                continue

            gt_data = load_yaml(gt_path)
            if int(image_id) not in gt_data:
                print(f"[SKIP] No GT entry for image {image_id} in class {class_id}")
                continue

            # Use the first instance for simplicity
            obj = gt_data[int(image_id)][0]
            R = obj['cam_R_m2c']  # List of 9 floats
            t = [val / 1000.0 for val in obj['cam_t_m2c']]  # Convert mm → meters

            # Write to CSV
            crop_path = os.path.join(CROP_FOLDER, filename)
            writer.writerow([crop_path, class_id, image_id] + t + R)

        except Exception as e:
            print(f"[ERROR] Processing {filename}: {e}")


[SKIP] No GT entry for image 1212 in class 08
[SKIP] No GT entry for image 1186 in class 06
[SKIP] No GT entry for image 1183 in class 06
[SKIP] No GT entry for image 1218 in class 08
[SKIP] No GT entry for image 1232 in class 08
[SKIP] No GT entry for image 1197 in class 08
[SKIP] No GT entry for image 1226 in class 08
[SKIP] No GT entry for image 1188 in class 08
[SKIP] No GT entry for image 1194 in class 08


In [8]:
import pandas as pd
import numpy as np

# Load the CSV
df = pd.read_csv('pose_labels.csv')

rows = []
for _, row in df.iterrows():
    # Extract full rotation matrix (3x3)
    R = np.array([
        row[['R1', 'R2', 'R3']].values,
        row[['R4', 'R5', 'R6']].values,
        row[['R7', 'R8', 'R9']].values
    ])

    # Use first two columns of R as 6D rotation representation
    r6d = R[:, :2].flatten()  # shape (6,)

    # Add everything to the new row
    rows.append([
        row['crop_path'],
        row['class_id'],
        row['image_id'],
        row['x'], row['y'], row['z'],
        *r6d
    ])

# Create new DataFrame
df_6d = pd.DataFrame(rows, columns=[
    'crop_path', 'class_id', 'image_id',
    'x', 'y', 'z',
    'r6d_1', 'r6d_2', 'r6d_3', 'r6d_4', 'r6d_5', 'r6d_6'
])

# Save to new CSV
df_6d.to_csv('pose_labels_6d.csv', index=False)
print("Saved pose_labels_6d.csv with 6D rotation representation.")

Saved pose_labels_6d.csv with 6D rotation representation.


In [9]:
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class Pose6DDataset(Dataset):
    def __init__(self, csv_path, image_size=224):
        self.data = pd.read_csv(csv_path)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        # Load and preprocess image
        image = Image.open(row['crop_path']).convert('RGB')
        image = self.transform(image)

        # Load target: 3 translation + 6 rotation values
        pose = row[['x', 'y', 'z',
                    'r6d_1', 'r6d_2', 'r6d_3',
                    'r6d_4', 'r6d_5', 'r6d_6']].values.astype('float32')
        pose = torch.tensor(pose)

        return image, pose


In [10]:
dataset = Pose6DDataset('pose_labels_6d.csv')
image, pose = dataset[0]
print("Image shape:", image.shape)  # [3, 224, 224]
print("Pose:", pose)                # Tensor with 9 values

Image shape: torch.Size([3, 224, 224])
Pose: tensor([ 0.0488,  0.0109,  0.7861,  0.0623, -0.9962, -0.6014,  0.0109,  0.7965,  0.0861])


In [11]:
import torch.nn as nn
import torchvision.models as models

class PoseRegression6D(nn.Module):
    def __init__(self, pretrained=True):
        super(PoseRegression6D, self).__init__()

        # Load pretrained ResNet18 and remove classifier
        resnet = models.resnet18(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # output: [B, 512, 1, 1]

        # Fully connected regression head
        self.head = nn.Sequential(
            nn.Flatten(),               # [B, 512]
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 9)           # Output: [x, y, z, r6d_1 .. r6d_6]
        )

    def forward(self, x):
        x = self.backbone(x)  # [B, 512, 1, 1]
        return self.head(x)   # [B, 9]


In [12]:
model = PoseRegression6D()
print(model)

/home/erythm/anaconda3/envs/6d/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/erythm/anaconda3/envs/6d/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


PoseRegression6D(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_s

In [16]:
import torch.optim as optim
from torch.utils.data import DataLoader

# Hyperparameters
BATCH_SIZE = 32
EPOCHS = 20
LEARNING_RATE = 1e-4
CSV_PATH = 'pose_labels_6d.csv'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class Pose6DDataset(Dataset):
    def __init__(self, csv_path, image_size=224):
        self.data = pd.read_csv(csv_path)
        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['crop_path']).convert('RGB')
        image = self.transform(image)

        pose = row[['x', 'y', 'z',
                    'r6d_1', 'r6d_2', 'r6d_3',
                    'r6d_4', 'r6d_5', 'r6d_6']].values.astype('float32')
        return image, torch.tensor(pose)

class PoseRegression6D(nn.Module):
    def __init__(self, pretrained=True):
        super(PoseRegression6D, self).__init__()

        resnet = models.resnet18(pretrained=pretrained)
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])  # output: [B, 512, 1, 1]

        self.head = nn.Sequential(
            nn.Flatten(),               # [B, 512]
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 9)           # Output: [x, y, z, r6d_1 .. r6d_6]
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.head(x)
        
# Dataset and Dataloader
dataset = Pose6DDataset(CSV_PATH)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# Model, Loss, Optimizer
model = PoseRegression6D().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# Training Loop
for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for images, poses in dataloader:
        images = images.to(DEVICE)
        poses = poses.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, poses)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(dataloader)
    print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {avg_loss:.6f}")

# Save model 
torch.save(model.state_dict(), 'pose_regression_6d.pth')
print("Model saved")

/home/erythm/anaconda3/envs/6d/lib/python3.13/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/erythm/anaconda3/envs/6d/lib/python3.13/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Epoch 1/20 — Loss: 0.200650
Epoch 2/20 — Loss: 0.083071
Epoch 3/20 — Loss: 0.041617
Epoch 4/20 — Loss: 0.027771
Epoch 5/20 — Loss: 0.019741
Epoch 6/20 — Loss: 0.016860
Epoch 7/20 — Loss: 0.014375
Epoch 8/20 — Loss: 0.012306
Epoch 9/20 — Loss: 0.011864
Epoch 10/20 — Loss: 0.013711
Epoch 11/20 — Loss: 0.010649
Epoch 12/20 — Loss: 0.009109
Epoch 13/20 — Loss: 0.008506
Epoch 14/20 — Loss: 0.008048
Epoch 15/20 — Loss: 0.007104
Epoch 16/20 — Loss: 0.007436
Epoch 17/20 — Loss: 0.006074
Epoch 18/20 — Loss: 0.008796
Epoch 19/20 — Loss: 0.007701
Epoch 20/20 — Loss: 0.006072
Model saved


In [15]:
from ultralytics import YOLO
import os
import cv2

# Paths
YOLO_MODEL = '../checkpoints/model_main.pt' 
LINEMOD_ROOT = '../dataset/Linemod_preprocessed/data/' 
OUTPUT_CROP_FOLDER = 'RGB_TestCrops' 
CROP_SIZE = (224, 224)
CLASS_ID = '15'
SKIP_CLASSES = ['03', '07'] 

os.makedirs(OUTPUT_CROP_FOLDER, exist_ok=True)
model = YOLO(YOLO_MODEL)

if CLASS_ID in SKIP_CLASSES:
    print(f"[SKIP] {CLASS_ID} is a missing class.")
else:
    class_path = os.path.join(LINEMOD_ROOT, CLASS_ID)
    rgb_folder = os.path.join(class_path, 'rgb')
    test_file = os.path.join(class_path, 'test.txt')

    if not os.path.exists(test_file):
        print(f"[SKIP] No test file in class {CLASS_ID}")
    else:
        with open(test_file, 'r') as f:
            image_ids = [line.strip() for line in f if line.strip()]

        print(f"[{CLASS_ID}] Processing {len(image_ids)} test images...")

        for img_id in image_ids:
            img_path = os.path.join(rgb_folder, f"{img_id}.png")
            image = cv2.imread(img_path)
            if image is None:
                print(f"[WARNING] Cannot read image: {img_path}")
                continue

            results = model(image)

            for det_idx, det in enumerate(results[0].boxes):
                class_idx = int(det.cls.item()) + 1
                if f"{class_idx:02d}" in SKIP_CLASSES:
                    continue

                x1, y1, x2, y2 = map(int, det.xyxy[0].tolist())
                x1, y1 = max(0, x1), max(0, y1)
                x2, y2 = min(image.shape[1], x2), min(image.shape[0], y2)

                cropped = image[y1:y2, x1:x2]
                cropped_resized = cv2.resize(cropped, CROP_SIZE)

                filename = f"cls{class_idx:02d}_img{img_id}_det{det_idx}.png"
                save_path = os.path.join(OUTPUT_CROP_FOLDER, filename)
                cv2.imwrite(save_path, cropped_resized)

                print(f" Saved: {filename}")

[15] Processing 1041 test images...

0: 480x640 1 phone, 64.5ms
Speed: 2.7ms preprocess, 64.5ms inference, 2.1ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0000_det0.png

0: 480x640 1 phone, 22.3ms
Speed: 3.7ms preprocess, 22.3ms inference, 5.3ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0001_det0.png

0: 480x640 1 phone, 19.6ms
Speed: 1.8ms preprocess, 19.6ms inference, 4.7ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0002_det0.png

0: 480x640 1 phone, 8.8ms
Speed: 1.9ms preprocess, 8.8ms inference, 2.0ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0003_det0.png

0: 480x640 1 phone, 8.3ms
Speed: 1.6ms preprocess, 8.3ms inference, 1.9ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0004_det0.png

0: 480x640 1 phone, 21.1ms
Speed: 1.6ms preprocess, 21.1ms inference, 4.9ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img0005_det0.png

0: 480x640 1 phone, 8.3ms
Speed

[ WARN:0@661.066] global loadsave.cpp:268 findDecoder imread_('../dataset/Linemod_preprocessed/data/15/rgb/1135.png'): can't open/read file: check file path/integrity



0: 480x640 1 phone, 14.1ms
Speed: 1.9ms preprocess, 14.1ms inference, 2.6ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1142_det0.png

0: 480x640 1 phone, 14.0ms
Speed: 1.5ms preprocess, 14.0ms inference, 2.2ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1143_det0.png

0: 480x640 1 phone, 19.0ms
Speed: 2.1ms preprocess, 19.0ms inference, 6.2ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1144_det0.png

0: 480x640 1 phone, 16.3ms
Speed: 1.7ms preprocess, 16.3ms inference, 6.4ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1145_det0.png

0: 480x640 1 phone, 13.7ms
Speed: 1.7ms preprocess, 13.7ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1146_det0.png

0: 480x640 1 phone, 13.7ms
Speed: 1.5ms preprocess, 13.7ms inference, 2.5ms postprocess per image at shape (1, 3, 480, 640)
 Saved: cls13_img1147_det0.png

0: 480x640 1 phone, 13.8ms
Speed: 2.0ms preprocess, 13.8ms infe

In [ ]:
import csv
import yaml
import numpy as np
import pandas as pd

CROP_FOLDER = 'RGB_TestCrops'
OUTPUT_CSV = 'pose_labels_test_6d.csv'

def load_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

rows = []

for filename in os.listdir(CROP_FOLDER):
    if not filename.endswith('.png'):
        continue

    try:
        parts = filename.split('_')
        class_id = parts[0][3:]      
        image_id = parts[1][3:]      

        gt_path = os.path.join(LINEMOD_ROOT, class_id, 'gt.yml')
        if not os.path.exists(gt_path):
            continue

        gt_data = load_yaml(gt_path)
        if int(image_id) not in gt_data:
            continue

        obj = gt_data[int(image_id)][0]
        R = np.array(obj['cam_R_m2c']).reshape(3, 3)
        t = [val / 1000.0 for val in obj['cam_t_m2c']]  # mm → meters

        # Convert to 6D rotation: use first two columns of R
        r6d = R[:, :2].flatten()  # shape (6,)

        row = [os.path.join(CROP_FOLDER, filename), class_id, image_id] + t + r6d.tolist()
        rows.append(row)

    except Exception as e:
        print(f"[ERROR] {filename}: {e}")

# Save final CSV
df = pd.DataFrame(rows, columns=[
    'crop_path', 'class_id', 'image_id',
    'x', 'y', 'z',
    'r6d_1', 'r6d_2', 'r6d_3', 'r6d_4', 'r6d_5', 'r6d_6'
])
df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {OUTPUT_CSV} with {len(df)} rows.") 
